In [3]:
from curl_cffi import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os

# Input & Output
input_file = r"D:\Nexensus_Projects\Incorporation\input_folder\Book1.xlsx"
output_file = r"D:\Nexensus_Projects\Incorporation\output_incorporation.csv"

df = pd.read_excel(input_file).drop_duplicates(subset=['cin'])[10675:]
print("total_input", len(df))

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.zaubacorp.com/"
}

# Create file with header if not exists
if not os.path.exists(output_file):
    pd.DataFrame(columns=["name", "cin", "Incorporation Date"]).to_csv(output_file, index=False)

for row in df.itertuples(index=False):
    cin = row.cin
    input_name = row.name

    print(f"Processing: {cin} | {input_name}")

    incorporation_date = None

    try:
        url = f"https://www.zaubacorp.com/companysearchresults/{cin}"
        session = requests.Session()

        response = session.get(
            url,
            headers=headers,
            impersonate="chrome",
            timeout=60
        )

        soup = BeautifulSoup(response.text, 'html.parser')
        company_info_div = soup.select_one("#company-information")

        if company_info_div:
            table = company_info_div.find("table")

            for tr in table.find_all("tr"):
                cols = tr.find_all("td")
                if len(cols) >= 2:
                    if cols[0].text.strip() == "Date of Incorporation":
                        incorporation_date = cols[1].text.strip()
                        break

    except Exception as e:
        print(f"Error for {cin}: {e}")

    # Fallback if not found
    if not incorporation_date:
        incorporation_date = ""

    # Save immediately (append mode)
    temp_df = pd.DataFrame([{
        "name": input_name,
        "cin": cin,
        "Incorporation Date": incorporation_date
    }])

    temp_df.to_csv(output_file, mode='a', header=False, index=False)

    print(f"Saved: {cin} → {incorporation_date}")

    # Delay (anti-blocking)
    time.sleep(random.randint(20, 30))

total_input 693
Processing: AAG-4796 | Naam Nalvar Business Solutions Llp
Saved: AAG-4796 → 2016-05-27
Processing: AAN-5386 | Yaanee Fashions Llp
Saved: AAN-5386 → 2018-11-13
Processing: L24239MH1994PLC079015 | Ducol Organics And Colours Limited
Saved: L24239MH1994PLC079015 → 
Processing: L52100MP2014PLC033570 | On Door Concepts Limited
Saved: L52100MP2014PLC033570 → 
Processing: L72200MH2004PLC144890 | Paramatrix Technologies Limited
Saved: L72200MH2004PLC144890 → 2004-03-08
Processing: ACQ-4064 | Taxzu Advisors Llp
Saved: ACQ-4064 → 
Processing: U55101MH2023PTC404342 | Honu Hospitalities Private Limited
Saved: U55101MH2023PTC404342 → 
Processing: ACL-7942 | Coldrock India Llp
Saved: ACL-7942 → 
Processing: U29100MH2008PTC183850 | Shanc Infratech Private Limited
Saved: U29100MH2008PTC183850 → 2008-06-23
Processing: U45100PB2017PTC047186 | Kay Jee Ess Automobiles Private Limited
Saved: U45100PB2017PTC047186 → 
Processing: U70109HR2021PTC100033 | Ngv Home Services Private Limited
Saved:

In [ ]:
############ Code to fetch data from company detail api of data.gov and store in a csv file ################
import requests
import pandas as pd
import os
import time
import random

file_name = "mca_data_4.csv"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.zaubacorp.com/"
}

def fetch_with_retry(url, max_retries=3):
    """Retry logic for unstable API (502/503 etc.)"""
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, headers=headers, timeout=120)

            if response.status_code == 200:
                return response

            print(f"⚠️ Attempt {attempt}: Status {response.status_code}")

        except requests.exceptions.RequestException as e:
            print(f"⚠️ Attempt {attempt}: Request failed - {e}")

        time.sleep(2 * attempt)  # exponential backoff

    return None


for i in range(1275000, 1500000, 1000):
    print(f"Offset: {i}")

    url = f"https://api.data.gov.in/resource/4dbe5667-7b6b-41d7-82af-211562424d9a?api-key=579b464db66ec23bdd000001f38b086979c44b5761104103f7643d36&format=json&offset={i}&limit=1000"

    response = fetch_with_retry(url)

    if not response:
        print("❌ Failed after retries, skipping...")
        continue

    # ✅ Check status
    if response.status_code != 200:
        print("❌ Bad response:", response.status_code)
        continue

    # ✅ Safe JSON parse
    try:
        data = response.json()
    except Exception:
        print("❌ Not JSON response")
        print(response.text[:200])  # debug
        continue

    records = data.get("records", [])

    if not records:
        print("No more data")
        break

    df = pd.DataFrame(records)

    df.to_csv(
        file_name,
        mode='a',
        header=not os.path.exists(file_name),
        index=False
    )

    time.sleep(random.randint(5,10))

Offset: 1275000
Offset: 1276000
Offset: 1277000
Offset: 1278000
Offset: 1279000
Offset: 1280000
Offset: 1281000
Offset: 1282000
Offset: 1283000
Offset: 1284000
Offset: 1285000
Offset: 1286000
Offset: 1287000
Offset: 1288000
Offset: 1289000
Offset: 1290000
Offset: 1291000
Offset: 1292000
Offset: 1293000
Offset: 1294000
Offset: 1295000
Offset: 1296000
Offset: 1297000
Offset: 1298000
Offset: 1299000
Offset: 1300000
Offset: 1301000
Offset: 1302000
Offset: 1303000
Offset: 1304000
Offset: 1305000
Offset: 1306000
Offset: 1307000
Offset: 1308000
Offset: 1309000
Offset: 1310000
Offset: 1311000
Offset: 1312000
Offset: 1313000
Offset: 1314000
Offset: 1315000
Offset: 1316000
Offset: 1317000
Offset: 1318000
Offset: 1319000
Offset: 1320000
Offset: 1321000
Offset: 1322000
Offset: 1323000
Offset: 1324000
Offset: 1325000
Offset: 1326000
Offset: 1327000
Offset: 1328000
Offset: 1329000
Offset: 1330000
Offset: 1331000
Offset: 1332000
Offset: 1333000
Offset: 1334000
Offset: 1335000
Offset: 1336000
Offset: 

In [19]:
import os
import json
import pandas as pd

input_folder = r"D:\Nexensus_Projects\Incorporation\output_folder\txt"
output_file = r"combined_output1.csv"

all_data = []

for file in os.listdir(input_folder):
    if file.endswith(".txt"):
        file_path = os.path.join(input_folder, file)

        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    record = json.loads(line.strip())

                    all_data.append({
                        "CIN": record.get("CIN"),
                        "CompanyName": record.get("CompanyName"),
                        "CompanyRegistrationdate_date": record.get("CompanyRegistrationdate_date")
                    })

                except json.JSONDecodeError:
                    print(f"❌ Skipping invalid line in {file}")

# 🔹 Create DataFrame
df = pd.DataFrame(all_data)

# 🔹 ✅ Remove duplicates based on CIN
df = df.drop_duplicates(subset=["CIN", "CompanyName", "CompanyRegistrationdate_date"], keep="first")

# 🔹 Save CSV
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"✅ Final CSV saved at: {output_file}")

✅ Final CSV saved at: combined_output1.csv


In [ ]:
############ Code to fetch data from company detail api of data.gov and store in a txt file ################
import requests
import os
import time
import random
import json

# 🔹 Base file name
BASE_FILE_NAME = "mca_data_3"
file_index = 1
file_name = f"{BASE_FILE_NAME}_{file_index}.txt"

# 🔹 File size limit (200 MB)
MAX_FILE_SIZE = 200 * 1024 * 1024

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.zaubacorp.com/"
}


def rotate_file_if_needed():
    """Rotate file if size exceeds limit"""
    global file_index, file_name

    if os.path.exists(file_name) and os.path.getsize(file_name) >= MAX_FILE_SIZE:
        file_index += 1
        file_name = f"{BASE_FILE_NAME}_{file_index}.txt"
        print(f"📂 Rotated to new file: {file_name}")


def write_to_txt(records):
    """Write records in JSON Lines format"""
    try:
        rotate_file_if_needed()

        with open(file_name, 'a', encoding='utf-8') as f:
            for record in records:
                json.dump(record, f, ensure_ascii=False)
                f.write('\n')

    except Exception as e:
        print(f"❌ Write Error: {e}")


def fetch_with_retry(url, max_retries=3):
    """Retry logic for unstable API (502/503 etc.)"""
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, headers=headers, timeout=120)

            if response.status_code == 200:
                return response

            print(f"⚠️ Attempt {attempt}: Status {response.status_code}")

        except requests.exceptions.RequestException as e:
            print(f"⚠️ Attempt {attempt}: Request failed - {e}")

        time.sleep(2 * attempt)  # exponential backoff

    return None


# 🔹 Main Loop
for i in range(0, 1501000, 1000):
    print(f"🚀 Offset: {i}")

    url = f"https://api.data.gov.in/resource/4dbe5667-7b6b-41d7-82af-211562424d9a?api-key=579b464db66ec23bdd000001f38b086979c44b5761104103f7643d36&format=json&offset={i}&limit=1000"

    response = fetch_with_retry(url)

    if not response:
        print("❌ Failed after retries, skipping...")
        continue

    # ✅ Safe JSON parse
    try:
        data = response.json()
    except Exception:
        print("❌ Not JSON response")
        print(response.text[:200])
        continue

    records = data.get("records", [])

    if not records:
        print("✅ No more data. Stopping.")
        break

    # ✅ Save to TXT (JSON Lines)
    write_to_txt(records)

    # 🔹 Random delay (anti-block)
    time.sleep(random.randint(5, 10))

🚀 Offset: 0
🚀 Offset: 1000
🚀 Offset: 2000
🚀 Offset: 3000
🚀 Offset: 4000
🚀 Offset: 5000
🚀 Offset: 6000
🚀 Offset: 7000
🚀 Offset: 8000
🚀 Offset: 9000
🚀 Offset: 10000
🚀 Offset: 11000
🚀 Offset: 12000
🚀 Offset: 13000
🚀 Offset: 14000
🚀 Offset: 15000
🚀 Offset: 16000
🚀 Offset: 17000
🚀 Offset: 18000
🚀 Offset: 19000
🚀 Offset: 20000
🚀 Offset: 21000
🚀 Offset: 22000
🚀 Offset: 23000
🚀 Offset: 24000
🚀 Offset: 25000
🚀 Offset: 26000
🚀 Offset: 27000
🚀 Offset: 28000
🚀 Offset: 29000
🚀 Offset: 30000
🚀 Offset: 31000
🚀 Offset: 32000
🚀 Offset: 33000
🚀 Offset: 34000
🚀 Offset: 35000
🚀 Offset: 36000
🚀 Offset: 37000
🚀 Offset: 38000
🚀 Offset: 39000
🚀 Offset: 40000
🚀 Offset: 41000
🚀 Offset: 42000
🚀 Offset: 43000
🚀 Offset: 44000
🚀 Offset: 45000
🚀 Offset: 46000
🚀 Offset: 47000
🚀 Offset: 48000
🚀 Offset: 49000
🚀 Offset: 50000
🚀 Offset: 51000
🚀 Offset: 52000
🚀 Offset: 53000
🚀 Offset: 54000
🚀 Offset: 55000
🚀 Offset: 56000
🚀 Offset: 57000
🚀 Offset: 58000
🚀 Offset: 59000
🚀 Offset: 60000
🚀 Offset: 61000
🚀 Offset: 62000
🚀 Off